# Why Not Just Use Python Lists?

Before scaling to distributed computing, we need to understand *why* standard Python data structures fall short for large-scale data. This notebook demonstrates the memory overhead of Python lists vs NumPy arrays — the foundation everything else builds on.

In [ ]:
import psutil
import sys
import numpy as np

print(f"Available RAM: {psutil.virtual_memory().available / (1024 ** 3):.1f} GB")

## Python Lists: Dynamic but Expensive

Python lists store *pointers* to objects, not raw data. Each integer is a full Python object (~28 bytes), plus an 8-byte pointer in the list. Lists also over-allocate memory to make `append()` fast (amortized O(1)).

In [ ]:
big_list = list(range(20_000_000))
print(f"List container size: {sys.getsizeof(big_list) / (1024 ** 2):.1f} MB")
print(f"(This measures only the pointer array, not the integer objects themselves!)")

### List Growth Pattern

Watch how memory jumps when Python needs to reallocate the pointer array:

In [ ]:
prev_size = sys.getsizeof(big_list)
for i in range(len(big_list)):
    big_list.append(i)
    new_size = sys.getsizeof(big_list)
    if new_size != prev_size:
        print(f"After {len(big_list):>12,} elements: {new_size / (1024**2):>8.1f} MB  (+{(new_size - prev_size) / 1024**2:.1f} MB)")
        prev_size = new_size

## NumPy Arrays: Contiguous and Efficient

NumPy stores raw data (e.g., 8-byte int64) in a contiguous memory block. No pointers, no Python object overhead per element. This also means better CPU cache utilization.

In [ ]:
big_array = np.arange(20_000_000, dtype=np.int64)
print(f"NumPy array size: {big_array.nbytes / (1024 ** 2):.1f} MB")
print(f"Python list container: {sys.getsizeof(big_list) / (1024 ** 2):.1f} MB")
print(f"\nNumPy uses {big_array.nbytes / sys.getsizeof(big_list) * 100:.0f}% of the list's pointer-array memory")
print(f"(And the list's integers consume additional memory on top of that!)")

### The `np.append()` Trap

Unlike Python lists, NumPy arrays have **fixed size**. Every `np.append()` allocates a *brand new* array and copies all data. Appending 100 elements to a 20M-element array copies ~150 MB × 100 times = **15 GB** of unnecessary memory traffic.

In [ ]:
small_array = np.arange(1_000_000)
%time _ = [np.append(small_array, i) for i in range(100)]
print("Each np.append() created a full copy of the array!")

## Summary

| Property | Python list | NumPy array |
|---|---|---|
| Storage | Pointers to objects | Contiguous raw data |
| Memory per int | ~36 bytes (pointer + object) | 8 bytes (int64) |
| Append | Fast (amortized O(1)) | Slow (full copy each time) |
| Vectorized math | No (Python loop) | Yes (C-speed operations) |
| CPU cache friendly | No (scattered objects) | Yes (contiguous memory) |

**Takeaway**: For large-scale data, always use NumPy (or Pandas/Dask which build on it). Never grow arrays with `np.append()` — pre-allocate instead.

## Exercise

**How much RAM would a 1-billion-element int64 NumPy array use?** Calculate it, then verify.

In [ ]:
# Exercise: calculate the expected size in GB, then create the array to verify

expected_gb = ___  # YOUR CALCULATION HERE: 1_000_000_000 elements * 8 bytes / ...
print(f"Expected: {expected_gb:.1f} GB")

# Uncomment to verify (only if you have enough RAM!):
# verify_array = np.zeros(1_000_000_000, dtype=np.int64)
# print(f"Actual:   {verify_array.nbytes / (1024**3):.1f} GB")
# del verify_array  # free memory immediately